# Experiment B — Learn a shared, structured microscopy energy

**Frozen pretrained U-Net → probability-domain anchored PGD → learned structured energy → 8-connected nucleus evaluation.**

This extends the fixed experiment without changing its MM potential or geometric filter shapes. The two existing strengths are learned from **0.5 each**, capped at **1.5**. Three additional learned features are zero-DC and positive-sum-normalized (select 2–4), with small bounded positive strengths. All parameters are **shared across iterations**.

The training loss reproduces the supplied U-Net **class-weighted CE (1,2,1) with spatial multiplier 1+2m** plus **1.5× class-weighted soft Dice (0.2,0.4,0.4)**. No differentiation through connected-component analysis; instance F1, merge/split, and Dice are used for held-out validation.

**Important:** Your checkpoint is not included. Make sure GPU is selected in Colab. This notebook uses a separate ZIP of Python modules; no need to run the older fixed-energy notebook.

## 1. Install dependencies and upload the experiment package

In [ ]:
%pip -q install psfmodels scipy scikit-image numba matplotlib pandas

In [ ]:
from google.colab import drive, files
drive.mount('/content/drive')
uploaded = files.upload()  # Upload experiment_b_colab_bundle.zip
from pathlib import Path
import zipfile, sys
bundle = next((Path(s) for s in uploaded if s.endswith('.zip')), None)
if bundle is None: raise RuntimeError('Upload experiment_b_colab_bundle.zip')
with zipfile.ZipFile(bundle) as zf: zf.extractall('/content')
SRC = Path('/content/experiment_b_colab')
assert (SRC / 'train_experiment_b.py').exists()
sys.path.insert(0, str(SRC))
print('Code:', SRC)

## 2. Choose the pretrained U-Net checkpoint

The loader strictly checks U-Net state-dict compatibility. If the checkpoint directory contains multiple files, select a specific `.pt` file.

In [ ]:
from unet_inference import locate_checkpoint
CHECKPOINT = Path('/content/drive/MyDrive/Colab Notebooks/proj_26/checkpoints_unet_baseline_v1_boosted_ce')
if CHECKPOINT.is_dir():
    candidates = sorted(x for x in CHECKPOINT.rglob('*') if x.suffix in ('.pt','.pth','.ckpt','.bin'))
    print('Available checkpoints:')
    for c in candidates: print(' ', c)
    if len(candidates) > 1:
        print('Set CHECKPOINT to the exact file you want, e.g. CHECKPOINT = CHECKPOINT / "latest.pt"')
print('Selected:', CHECKPOINT)

## 3. Configuration

**Pilot:** 2 training volumes, 1 validation volume, ~10 updates, short horizons. **Full:** raise volumes/updates/validation slices and turn on the long warmups. Training data and validation data have nonoverlapping seed ranges. This does not certify independence from your pretrained U-Net's training set.

The existing fixed energy geometry and coefficients below match your last provided configuration. The first energy step uses `step_size=0.02`, `anchor_mu=0.4`; projected descent uses your original reduced-coordinate Euclidean projection.

**SGD note:** This is vanilla SGD (`momentum=0`), with LR `1e-3` and weight decay `1e-5`. Because this is higher-order differentiation, the gradients may be small; the CSV logs let you check that before increasing the learning rate.

In [ ]:
import torch
from train_experiment_b import TrainBConfig, train
cfg = TrainBConfig(
    checkpoint=str(CHECKPOINT),
    output_dir='/content/drive/MyDrive/Colab Notebooks/proj_26/experiment_b_results',
    device='cuda' if torch.cuda.is_available() else 'cpu',
    seed=2400, train_seed=200000, val_seed=300000,
    train_volumes=8, val_volumes=3, N=192, z_slices=16,
    z_ratio=2, n_seeds=45, class_probs=(2/5,1/5,2/5),
    train_domain='a', val_domain='a',
    n_extra=3, extra_filter_size=13,
    strength_cap=1.5, extra_strength_cap=0.3,
    kernel_raw_cap=2., initial_extra_strength=.002,
    anchor_mu=.4, step_size=.02,
    updates=250, learning_rate=1e-3, momentum=0., weight_decay=1e-5,
    batch_size=2, warmups=(0,20,80,160,300), differentiable_steps=4,
    source_mode='mixed', prob_unet_in_mixed=.5,  # 'unet' / 'corrupted' also work
    gt_blur_range=(0.,1.5), gt_spectral_range=(.4,1.2),
    restart_probability=.5, restart_blur_range=(0.,.8), restart_spectral_range=(0.,.5),
    eval_every=25, val_horizons=(0,50,150,300,400),
    validation_max_slices=8,
    energy=dict(
        lam=.2, epsilon=.5, well_cubic_weight=1.,
        barrier_weight=.5, barrier_width=5, nuclear_band_width=2,
        barrier_length=13, barrier_longitudinal_sigma=3.5,
        barrier_orientations=8,
        boundary_weight=.5, boundary_middle_width=5,
        boundary_outer_width=2, boundary_orientations=16,
        learn_strengths=False, learn_residuals=False,
    ),
)
print('Device:', cfg.device)

## 4. Optional quick pilot (recommended first)

This uses a **separate pilot directory**, leaving the full configuration unchanged.

In [ ]:
from dataclasses import replace
pilot = replace(cfg, train_volumes=1, val_volumes=1, updates=3,
                warmups=(0,2), val_horizons=(0,2,5),
                validation_max_slices=2, eval_every=2,
                output_dir=str(Path(cfg.output_dir) / 'pilot'))
# Uncomment to check checkpoint compatibility, data generation and gradients:
# pilot_model, pilot_history, pilot_validation = train(pilot)

## 5. Train experiment B

Captures training histories and checkpoints to Google Drive. This is computationally expensive: increase training/validation sizes only after the pilot succeeds.

In [ ]:
model, history, validation_history = train(cfg)

## 6. Plot training dynamics and held-out instance metrics

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
out=Path(cfg.output_dir)
tr=pd.read_csv(out/'train_history.csv')
va=pd.read_csv(out/'validation_history.csv')
fig,axes=plt.subplots(1,3,figsize=(16,4))
axes[0].plot(tr['update'],tr['loss']); axes[0].set(title='Training objective',xlabel='Update',ylabel='Loss')
for k in ['separator_strength','boundary_strength']+[f'extra_strength_{i}' for i in range(cfg.n_extra)]:
    axes[1].plot(tr['update'],tr[k],label=k)
axes[1].legend(fontsize=7); axes[1].set(title='Learned filter strengths',xlabel='Update')
for t in sorted(va['iteration'].unique()):
    d=va[va['iteration']==t]
    axes[2].plot(d['update'],d['micro_instance_f1'],label=f'{t} iterations')
axes[2].legend(fontsize=8); axes[2].set(title='Held-out instance micro F1',xlabel='Update',ylabel='F1')
fig.tight_layout(); fig.savefig(out/'training_curves.png',dpi=150); plt.show()
print(va.groupby('iteration').tail(1).to_string(index=False))

## 7. Evaluate Domain B/C without retraining (optional)

The simplest method is to call `prepare_cache` and `evaluate` using the saved best checkpoint. The U-Net remains frozen and the learned energy is unchanged. Use a **new seed range** for final test domains and do not use their F1 values to choose hyperparameters.

In [ ]:
# Optional example; uncomment only after training.
# from train_experiment_b import prepare_cache, evaluate
# from unet_inference import load_unet
# net, _ = load_unet(str(CHECKPOINT), cfg.device)
# test_cache = prepare_cache(net, cfg, seed=400000, n_volumes=10, domain='b')
# from learned_energy import StructuredEnergy
# from mm_fixed_compatible import Config
# best = torch.load(Path(cfg.output_dir)/'best.pt',map_location=cfg.device,weights_only=False)
# saved_tc = best['training_config']
# learned = StructuredEnergy(Config(**best['model_config']),cfg.n_extra,cfg.extra_filter_size,
#      cfg.strength_cap,cfg.extra_strength_cap,cfg.kernel_raw_cap,cfg.initial_extra_strength,cfg.seed).to(cfg.device)
# learned.load_state_dict(best['model_state_dict'])
# per_slice, summary = evaluate(learned,test_cache,cfg,max_slices=None)
# print(summary)

## Design notes

- `learned_energy.py`: capped strengths and normalized additional feature filters, original reduced-coordinate PGD and full-three-class anchor.
- `experiment_b_loss.py`: exact supplied supervised CE + Dice loss, evaluated on probabilities.
- `train_experiment_b.py`: optional initial and restart corruption, frozen U-Net prediction cache, random detached warmups, 4-step differentiable rollout, SGD, checkpointing, held-out evaluation.
- `instance_metrics.py`: 8-connected nuclear instance F1@0.5, merges, splits, PQ and class Dice.

**Deliberate simplifications:** The additional filters have no rotational tying; each is an independent spatial feature. Their boundedness is guaranteed through normalized kernels and bounded scalar strengths, *not* by separately capping the coefficients of the final normalized kernels. Validation F1 is discrete and is not optimized directly; training uses weighted CE + soft Dice. Different source types use the same anchor relative to their own initial prediction. Original U-Net weights are never updated.